# 12 - Fast animated dashboard

In [ ]:
import os,html
CATALOG=os.getenv('AIDP_CATALOG','aidp_poc')
m=spark.table(f'{CATALOG}.gold.dashboard_metrics').first()
bronze=spark.table(f'{CATALOG}.gold.dashboard_bronze_categories').orderBy('events',ascending=False).collect()
segments=spark.table(f'{CATALOG}.gold.dashboard_segments').orderBy('kwh',ascending=False).collect()
preds=spark.table(f'{CATALOG}.gold.dashboard_predictions').limit(100).collect()
def table(items,cols):
 h=''.join('<th>'+label+'</th>' for _,label in cols); b=''.join('<tr>'+''.join('<td>'+html.escape(str(getattr(x,key,None) if getattr(x,key,None) is not None else '-'))+'</td>' for key,_ in cols)+'</tr>' for x in items) or '<tr><td colspan=9>No data</td></tr>'; return '<table><thead><tr>'+h+'</tr></thead><tbody>'+b+'</tbody></table>'
css='''<style>.dash{font-family:Arial;color:#fff;background:radial-gradient(circle at 10% 0,#12366d,#06152d 55%,#020812);padding:26px;border-radius:18px}.dash h1{margin:0;font-size:29px;color:#fff}.sub{color:#d6e6ff;font-size:13px;margin:7px 0 20px}.live{float:right;background:#0e5c51;color:#fff;border-radius:20px;padding:9px 14px;font-size:12px;font-weight:bold}.dot{display:inline-block;width:9px;height:9px;border-radius:50%;background:#7fffd4;margin-right:7px;animation:p 1s infinite}.grid{display:grid;grid-template-columns:repeat(4,1fr);gap:13px}.card{background:#0d2854;border:1px solid #5693db;border-radius:13px;padding:16px;box-shadow:0 12px 25px #0008;transition:.25s}.card:hover{transform:translateY(-6px) perspective(700px) rotateX(3deg)}.label{color:#d5e7ff;font-size:11px;letter-spacing:1px}.value{font-size:27px;font-weight:800;color:#fff;margin-top:7px;text-shadow:0 0 14px #62caff}.panel{margin-top:15px;background:#071c3d;border:1px solid #3f78bb;border-radius:13px;padding:16px}table{width:100%;border-collapse:collapse;margin-top:8px}th{text-align:left;color:#cfe4ff;font-size:11px;padding:9px 5px}td{color:#fff;padding:10px 5px;border-top:1px solid #ffffff2b}td:not(:first-child),th:not(:first-child){text-align:right}@keyframes p{50%{transform:scale(1.8);opacity:.2}}@media(max-width:700px){.grid{grid-template-columns:repeat(2,1fr)}}</style>'''
page=css+f'''<div class=dash><div class=live><span class=dot></span>READY {html.escape(str(m.refreshed_at))}</div><h1>GridPulse Fast Command Center</h1><div class=sub>Pre-aggregated Spark metrics high-contrast dark navy</div><div class=grid><div class=card><div class=label>BRONZE EVENTS</div><div class=value>{m.bronze_events:,}</div></div><div class=card><div class=label>BRONZE ACTIVE METERS</div><div class=value>{m.bronze_active_meters:,}</div></div><div class=card><div class=label>SILVER READINGS</div><div class=value>{m.silver_readings:,}</div></div><div class=card><div class=label>PEAK DEMAND</div><div class=value>{m.peak_kw or 0} kW</div></div></div><div class=panel><b>Bronze events and active meters by meter type and category</b>{table(bronze,[('meter_type','Meter type'),('customer_segment','Customer category'),('events','Events'),('active_meters','Active meters')])}</div><div class=panel><b>Gold energy by segment</b>{table(segments,[('customer_segment','Segment'),('meters','Meters'),('kwh','Energy kWh'),('peak_kw','Peak kW')])}</div><div class=panel><b>Latest ML forecasts 100 meters {html.escape(str(m.model_version))}</b>{table(preds,[('meter_id','Meter'),('prediction_kwh','Forecast kWh'),('interval_start_utc','Interval')])}</div></div>'''
class View:
 def __init__(self,x): self.x=x
 def _repr_html_(self): return self.x
 def __repr__(self): return 'HTML renderer unavailable.'
View(page)